> ### EEE3314: Introduction to Artificial Intelligence

# Assignment II: Ensemble

### *Assignement Instructions:*
- **Due date**: 11:55 PM, Nov. 22 (Fri), 2024
- **File name**: [ID_Name_HW2.ipynb]
- Complete project2.py with comments describing your implementation after reading instructions in this file.
- **Respond to all the questions that requires you to answer with markdown, theoretically and empirically**

In [ ]:
import datetime
print("This code is written at " + str(datetime.datetime.now()))

In [ ]:
%load_ext autoreload
%autoreload 2

## Problem statement

We will use datasets generated from ```sklearn.dataset.make_classification``` and ```sklearn.dataset.make_blobs```. Let's first viaualize the $\texttt{classification}$ and $\texttt{blobs}$ datasets with the ```scatter``` function from ```matplotlib.pyplot```.

**Prepare Classification dataset**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_classification

cm_bright = ListedColormap(['#FF0000', '#0000FF'])
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2, random_state=10, n_clusters_per_class=2)
plt.scatter(X[:,0], X[:,1], s=40, c=y, cmap=cm_bright, edgecolors='k')
plt.show()

**Prepare Blobs dataset**

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=400, centers=4, cluster_std=2.5, random_state=13)
y[y==1]=0; y[y==2]=1; y[y==3]=1
y[y == 0] = -1

cm_bright = ListedColormap(['#FF00FF', '#00FF00'])
plt.scatter(X[:,0], X[:,1], s=40, c=y, cmap=cm_bright, edgecolors='k')
plt.show()

We will implement the $\texttt{RandomForest}$ and $\texttt{AdaBoost}$ algorithms. In the case of $\texttt{RandomForest}$, **we will implement a simple version, not the complete version**. 

Additionally, we will analyze the characteristics of each algorithm using $\texttt{Precision}$, $\texttt{Recall}$, $\texttt{F1 score}$, and $\texttt{Precision-Recall curve}$.

These two algorithms differ in numerous aspects. Let's apply $\texttt{SimpleRandomForest}$ and $\texttt{AdaBoost}$ to the datasets and find how they are different.

---

### DecisionTreeClassifier ###

We will use ```sklearn.tree.DecisionTreeClassifier``` to easily implement $\texttt{SimpleRandomForest}$ and $\texttt{AdaBoost}$.
```python
from sklearn.tree import DecisionTreeClassifier
```

* **SimpleRandomForest**
    * Use ```DecisionTreeClassifier(max_depth=self.max_depth)``` as a decision tree.
    
      
* **AdaBoost**
    * Use ```DecisionTreeClassifier(max_depth=1)``` as a decision stump.
    
    
In this assignment, we will use $\texttt{DecisionTreeClassifier}$ to implement $\texttt{SimpleRandomForest}$ and $\texttt{AdaBoost}$.

> ```DecisionTreeClassifier.fit(X, y, sample_weight=None)```
> ```
>Description:
>    Build a decision tree classifier from the training set.
> Args:
>     X: (N, d) numpy array. Training samples.
>     y: (N,) numpy array. Ground truth classes.
>     sample_weight: [Option] (N,) numpy array. Weights for sampling with replacement.
> Returns:
> ```

> ```DecisionTreeClassifier.predict(X)```
>```
>Description:
>    Predict classes for X.
>Args:
>    X: (N, d) numpy array. Training/testing samples.            
>Returns:
>    pred: (N, ) numpy array. Prediction of classifier.
>```

**NOTE**: 
- In $\texttt{SimpleRandomForest}$ and $\texttt{AdaBoost}$ implementation, exploiting ```sklearn.tree``` and ```sklearn.ensemble``` (or equivalent) except ```sklearn.tree.DecisionTreeClassifier``` **will not be allowed**.
  
- When you use ```sklearn.tree.DecisionTreeClassifier```, just set the parameter mentioned above, and use the default setting for other parameters.

---

### P1. "Simple" RandomForest (*20 points*)

P1.1 Implement a simple random forest (```SimpleRandomForest``` class) for binary classification. Note that the RandomForest algorithm uses a **Bagging** (i.e. Bootstrap aggregation) and a randomization technique. (*10 points*)  

> **Bagging and randomization**: RandomForest begins by creating multiple bootstrap samples from the original dataset ${D_1, D_2, ..., D_t}$. Each sample is generated by randomly selecting data points with replacement. Furthermore, best split is chosen from a random sample of attributes instead of all attributes.
>> * For each decision tree in the **original** RandomForest, a random subset of dataset and a random subset of features are chosen at **each node** to determine the best split.  
>> * <U>However, we will use randomized features for ***each decision tree***, not for each node.<U>  
>> * That is, you can just train ```DecisionTreeClassifier()``` class (i.e. a decision tree) on **a single set of randomized features** of bootstrap samples.
>  
> **Quality metric**: Classification error computing the fraction of mistakes.  
> **Prediction**: $$\hat{\mathbf{y}}=\text{sign} \left( \sum_{t=1}^{T} f_t(\mathbf{x}(D_t[:, \text{indices_t}])) \right)$$(voting)  
> **Save weak classifier**: Use ```dictionary```, i.e., {```'feature_indices'```: $\text{indices_t}$, ```'classifier'```: $f_t(\mathbf{x})$}. Then append the dictionary to the attribute list (```self.classifiers```).  
    
*Refer the lecture note 12 p.11-16.   

P1.2 Run the codes provied to plot accuracies over the number of classifiers and max_depth. Analyze the results and how each component affects the performance. (*5 points*) 


P1.3 Implement the ```compute_each_accuracies``` function that inputs classifiers of bagging (i.e., a list of dictionary that contains 'feature_indices' and 'classifier' as keys), a training dataset, a test dataset, and outputs the prediction accuracies on training and test datasets. Plot the accuracies on training and testing samples of each classifier. Use the $\frac{\text{number of correct predictions}}{\text{total number of samples}}$ to compute the accuracy. (*5 points*)

### P2. AdaBoost (*20 points*)

P2.1 Implement an adaboost (```AdaBoost``` class) for binary classification. Note that the AdaBoost algorithm uses a **weighted sampling with replacement** to learn the weak classifier $f_{t}(\mathbf{x})$ with data weights $\alpha$. (*10 points*)  
> **Weighted sampling with data weight**: This samples data points from the original dataset with corresponding sampling probabilities (i.e., weights for individual training samples).
>> For example, let's suppose the following data points $\mathbf{x}$ and corresponding weights $\alpha$: $\mathbf{x}=[x_1, x_2, x_3]$ and $\alpha = [0.1, 0.6, 0.3]$. In the weighted sampling procedure, $x_1$, $x_2$, and $x_3$ are chosen with probabilities of $0.1$, $0.6$, and $0.3$, respectively. That is, the data point with a higher weight is more likely to be chosen.
>  
> **Quality metric**: Classification error computing the fraction of mistakes.  
> **Compute coefficient**: $\hat{w}_t = \frac{1}{2} \ln \left( \frac{1 - \epsilon_t}{\epsilon_t} \right)$  
> **Recompute weight**: $\alpha_{t+1}^{i} = \alpha_{t}^{i} e^{{-\hat{w}_{t} y_{i} f_{t}(\mathbf{x})}}$ , where $y_{i} = 1 \text{or} -1$.  
> **Normalize weight**: $\alpha_{t}^{i} = \frac{\alpha_{t}^{i}}{\sum_{i=1}^{N} \alpha_{t}^{i}}$  
> **Prediction**: $\hat{\mathbf{y}}=\text{sign} \left( \sum_{t=1}^{T} \hat{w_t} f_t(\mathbf{x}) \right)$  
> **Save weak classifier**: Use ```dictionary```, i.e., {```'coefficient'```: $\hat{w}_t$, ```'classifier'```: $f_t(\mathbf{x})$}. Then append the dictionary to the attribute list (```self.classifiers```).  

*Refer the lecture note 12 p.35-37.*

P2.2 Run the code provied to plot the weighted error and the coefficient of each classifier. Analyze the results and relationship between the weighted errors and the coefficients. (*5 points*)  

P2.3 Implement the ```compute_staged_accuracies``` function that inputs boosted classifiers (i.e., a list of dictionary that contains 'coefficient' and 'classifier' as keys), a training dataset, a test dataset, and outputs the prediction accuracies on training and test datasets. Plot the accuracies on training and testing samples, over the number of classiferes. Use the $\frac{\text{number of correct predictions}}{\text{total number of samples}}$ to compute the accuracy. (*5 points*)



### P3 Precision and Recall (*20 points*)

P3.1 Implement the ```get_precision```, ```get_recall```, ```get_f1_score``` function (*10 points*)

P3.2 Implement the ```get_precision_recall_curve``` funcrtion, using probabilities of positive class in y_proba as thresholds in ```get_precision_recall_curve```
(x-axis should be *recall* and y-axis should be *precision*) (*10 points*)

### P4. Analysis (*40 points*)

P4.1 Describe the differences between bagging and boosting with your observations from ```SimpleRandomForest``` and ```AdaBoost```. (*10 points*)

P4.2 Analyze and contrast the logistic regression, (simple) randomforest, and adaboost in terms of objective functions. (*5 points*)

P4.3    
(1) Analyze and contrast the logistic regression, (simple) random forest, and adaboost based on the *Precision*, *Recall* and *F1 score*. (*5 points*)\
(2) Analyze and contrast the logistic regression, (simple) random forest, and adaboost based on the *Precision-Recall* curve. (*5 points*)

P4.4  
(1) Run the codes provided to plot decision boundaries and prediction accuracies on test dataset. Analyze and contrast the decision boundaries of the classification methods (logistic regression, bagging decision tree, and boosted decision tree). (*10 points*)  
(2) Analyze and contrast the decision boundaries of the classification methods in the aspect of overfitting. Note that we are using more training samples, compared to the previous problem of P4.3(1).   (*5 points*)

---

## Results

In [ ]:
from copy import deepcopy

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

import sklearn.datasets
from sklearn.model_selection import train_test_split

from project2 import *

### Prepare Dataset

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=2000, n_features=15, n_redundant=0, n_informative=12, random_state=1, n_clusters_per_class=2, weights=[0.5])
y[y == 0] = -1

n_split = 1500

X_train, X_test = X[:n_split], X[n_split:]
y_train, y_test = y[:n_split], y[n_split:]

print(X_train.shape)
print(X_test.shape)

---

#### P1.1 Implement a simple random forest (```SimpleRandomForest``` class) for binary classification. Note that the RandomForest algorithm uses a **Bagging** (i.e. Bootstrap aggregation) and a randomization technique. (*10 points*)  

*Refer the lecture note 12 p.11-16.   

In [ ]:
randomforest_classifier = SimpleRandomForest(50, max_depth=5)
randomforest_classifier.fit(X_train, y_train)

print(f"train: {np.average(randomforest_classifier.predict(X_train) == y_train)}, test: {np.average(randomforest_classifier.predict(X_test) == y_test)}")

In [ ]:
classifiers = randomforest_classifier.classifiers
print(len(classifiers))
print(classifiers[0].keys())
print(classifiers[0].values())

---

#### P1.2 Run the codes provied to plot accuracies over the number of classifiers and max_depth. Analyze the results and how each component affects the performance. (*5 points*) 

Answer:
Increasing the max_depth parameter generally leads to a more complex model, which can reduce bias effectively. This is observed as both training and test accuracies increase with the depth of the trees. However, significantly higher values of max_depth may lead to overfitting, which is evident from an increase in variance. Essentially, while deeper trees capture more detailed information, they also tend to learn too much from the training data, potentially decreasing the model's generalization to new data. On the other hand, augmenting the number of classifiers typically helps in mitigating variance. The rationale behind this is that having more classifiers in the ensemble allows for averaging out the predictions across a larger set of decision trees, smoothing out fluctuations and leading to more stable predictions. Consequently, an increase in the number of classifiers usually enhances the accuracy of the model, as the ensemble becomes more robust against individual tree errors.

In [ ]:
train_accuracy_estimators = []
test_accuracy_estimators = []
num_estimator_list = [i for i in range(1,21)]

for num_estimator in num_estimator_list:
    randomforest_classifier = SimpleRandomForest(num_estimator, max_depth=5)
    randomforest_classifier.fit(X_train, y_train)
    acc_train = np.average(randomforest_classifier.predict(X_train) == y_train)
    acc_test = np.average(randomforest_classifier.predict(X_test) == y_test)
    train_accuracy_estimators.append(acc_train)
    test_accuracy_estimators.append(acc_test)
    

plt.figure(figsize=[15, 8])

plt.plot(num_estimator_list, train_accuracy_estimators, label='train')
plt.plot(num_estimator_list, test_accuracy_estimators, label='test')
plt.title('Accuracies over # of classifiers')
plt.xlabel('# of classifiers')
plt.ylabel('accuracy')
plt.legend()
plt.show()

In [ ]:
train_accuracy_depth = []
test_accuracy_depth = []
depth_list = [i for i in range(1,11)]

for depth in depth_list:
    randomforest_classifier = SimpleRandomForest(50, max_depth=depth)
    randomforest_classifier.fit(X_train, y_train)
    acc_train = np.average(randomforest_classifier.predict(X_train) == y_train)
    acc_test = np.average(randomforest_classifier.predict(X_test) == y_test)
    train_accuracy_depth.append(acc_train)
    test_accuracy_depth.append(acc_test)

plt.figure(figsize=[15, 8])

plt.plot(depth_list, train_accuracy_depth, label='train')
plt.plot(depth_list, test_accuracy_depth, label='test')
plt.title('Accuracies over max_depth')
plt.xlabel('# of max_depth')
plt.ylabel('accuracy')
plt.legend()
plt.show()

---

#### P1.3 Implement the ```compute_each_accuracies``` function that inputs classifiers of bagging (i.e., a list of dictionary that contains 'feature_indices' and 'classifier' as keys), a training dataset, a test dataset, and outputs the prediction accuracies on training and test datasets. Plot the accuracies on training and testing samples of each classifier. Use the $\frac{\text{number of correct predictions}}{\text{total number of samples}}$ to compute the accuracy. (*5 points*)

In [ ]:
classifiers = randomforest_classifier.classifiers
acc_train, acc_test = compute_each_accuracies(classifiers, X_train, y_train, X_test, y_test)

In [ ]:
plt.figure(figsize=[15, 8])

plt.stem(acc_train, label='train')
plt.stem(acc_test, 'tab:orange', label='test')
plt.title('Accuracies over Index of classifier')
plt.xlabel('Index of classifier')
plt.ylabel('accuracy')
plt.ylim(0,1)

plt.legend(loc='lower right')
plt.show()

---

#### P2.1 Implement an adaboost (```AdaBoost``` class) for binary classification. Note that the AdaBoost algorithm uses a **weighted sampling with replacement** to learn the weak classifier $f_{t}(\mathbf{x})$ with data weights $\alpha$. (*10 points*)

In [ ]:
adaboost_classifier = AdaBoost(50)
adaboost_classifier.fit(X_train, y_train)

print(f"train: {np.average(adaboost_classifier.predict(X_train) == y_train)}, test: {np.average(adaboost_classifier.predict(X_test) == y_test)}")

In [ ]:
classifiers = adaboost_classifier.classifiers
print(len(classifiers))
print(classifiers[0].keys())
print(classifiers[0].values())

In [ ]:
adaboost_classifier.error_history

In [ ]:
coefficients = [classifier['coefficient'] for classifier in adaboost_classifier.classifiers]
coefficients

In [ ]:
adaboost_classifier.data_weight

---

#### P2.2 Run the code provied to plot the weighted error and the coefficient of each classifier. Analyze the results and relationship between the weighted errors and the coefficients. (*5 points*)  

Answer:
The AdaBoost algorithm fundamentally links the classifier coefficient inversely with the weighted error. Specifically, as the weighted error of a classifier decreases, its coefficient increases, enhancing its influence on the final model prediction. Conversely, if the weighted error increases, the corresponding classifier's coefficient is reduced. This mechanism is central to AdaBoost’s strategy, which effectively amplifies the impact of more accurate classifiers within the ensemble. By doing so, AdaBoost optimizes the overall predictive performance by focusing more on classifiers that perform well, thereby assigning them greater decision-making power in the aggregated output.

In [ ]:
plt.figure(figsize=[7, 5])

plt.stem(range(1, len(adaboost_classifier.error_history)+1), adaboost_classifier.error_history)
plt.title('Weighted error of each classifier')
plt.xlabel('# of classifier')
plt.ylabel('error')
plt.show()

In [ ]:
plt.figure(figsize=[7, 5])

plt.stem(range(1, len(coefficients)+1), coefficients)
plt.title('Coefficient of each classifier')
plt.xlabel('classifier index')
plt.ylabel('coefficient')
plt.show()

In [ ]:
coeff = np.array(coefficients)

In [ ]:
coeff_top5 = np.argsort(coeff)[::-1][:5]
coeff_bottom5 = np.argsort(coeff)[::-1][-5:]

In [ ]:
coeff_bottom5

---

#### P2.3 Implement the ```compute_staged_accuracies``` function that inputs boosted classifiers (i.e., a list of dictionary that contains 'coefficient' and 'classifier' as keys), a training dataset, a test dataset, and outputs the prediction accuracies on training and test datasets. Plot the accuracies on training and testing samples, over the number of classiferes. Use the $\frac{\text{number of correct predictions}}{\text{total number of samples}}$ to compute the accuracy. (*5 points*)

In [ ]:
classifiers = adaboost_classifier.classifiers    

acc_train, acc_test = compute_staged_accuracies(classifiers, X_train, y_train, X_test, y_test)

In [ ]:
plt.figure(figsize=[15, 8])

plt.plot(acc_train, label='train')
plt.plot(acc_test, label='test')
plt.title('Accuracies over # of classifiers')
plt.xlabel('# of classifiers')
plt.ylabel('accuracy')
plt.legend()
plt.show()

---

#### P3.1 Implement the ```get_precision```, ```get_recall```, ```get_f1_score``` function (*10 points*)

In [ ]:
y_pred = randomforest_classifier.predict(X_test)

precision = get_precision(y_pred, y_test)
recall = get_recall(y_pred, y_test)
f1_score = get_f1_score(y_pred, y_test)

print(f"precision: {precision}, recall: {recall}, f1_score: {f1_score}")

#### P3.2 Implement the ```get_precision_recall_curve``` funcrtion, using probabilities of positive class in y_proba as thresholds in ```get_precision_recall_curve```
(x-axis should be *recall* and y-axis should be *precision*) (*10 points*)

In [ ]:
pre_ran, rec_ran = plot_precision_recall_curve(randomforest_classifier.predict_proba(X_test), y_test)

# Plot the precision-recall curve
plt.figure(figsize=[9, 8])
plt.plot(pre_ran, rec_ran, marker='.', label='RandomForest')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

---

#### P4.1 Describe the differences between bagging and boosting with your observations from ```SimpleRandomForest``` and ```AdaBoost```. (*10 points*)

Answer:
Bagging (Bootstrap AGGregation) Bagging leverages the bootstrap resampling technique to introduce randomness into the model training process, helping to prevent overfitting. In the context of RandomForest models, each tree is trained on a bootstrap sample and constructed independently, which allows for parallel processing. Typically, during tree construction, a random subset of features is used at each split point instead of the full feature set. In this project, we've chosen to use a consistent set of randomized features for each tree. After building multiple trees, they are aggregated to form the final model where each tree contributes equally to the outcome. The main benefits of bagging include a reduction in variance and enhanced computational efficiency through parallelism. However, it's important to note that bagging primarily affects model variance and does not significantly alter the inherent bias of the base models.

Boosting, another powerful ensemble technique, focuses on sequentially training a series of weak learners, each addressing the errors of its predecessors. The process starts with weighted sampling of the data, where each instance's likelihood of selection is based on its associated weight. A weak classifier is then trained on this sample. Following training, the weighted error of the classifier is computed, which is then used to adjust the weights of the data instances for the subsequent classifier in the series. This adjustment helps in focusing the next classifier on the harder-to-classify instances. Each classifier's influence on the final prediction is determined by a coefficient derived from its error rate, allowing those with lower errors to have greater impact. Boosting is particularly effective at reducing bias, as each successive learner works to correct errors from the previous models. However, due to its sequential nature, Boosting can be slower than other parallelizable techniques, making it computationally intensive in some scenarios.

---

#### P4.2 Analyze and contrast the logistic regression, (simple) randomforest, and adaboost in terms of objective functions. (*5 points*)

Answer:
Logistic Regression utilizes the cross-entropy loss as its objective function, with the primary goal being to minimize this loss across the dataset. This function measures the performance of the classification model whose output is a probability value between 0 and 1.

Random Forest, in contrast to Logistic Regression, does not operate based on a singular, explicitly defined objective function. Instead, it constructs each tree in the ensemble independently, aiming to minimize the classification error for each tree. The ensemble result is then obtained through a process called bagging, where the individual predictions are aggregated to produce a final outcome. Although it doesn't have a consolidated objective function like some models, the goal of reducing classification error in each tree guides the construction process.

AdaBoost's approach is to minimize an exponential loss function, which directly targets the classification errors. It achieves this by focusing on reducing the errors of weak learners that perform just above chance level. Each weak classifier contributes to the final model by improving on the areas where previous classifiers were deficient. If each classifier's error rate is less than 0.5, AdaBoost can effectively reduce the cumulative training error to zero, capitalizing on the exponential nature of its loss function.

**Logisitc Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

regrssor = LogisticRegression()
regrssor.fit(X_train, y_train)
print(f"train: {np.average(regrssor.predict(X_train) == y_train)}, test: {np.average(regrssor.predict(X_test) == y_test)}")

**Decision Stump**

In [ ]:
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)
print(f"train: {np.average(stump.predict(X_train) == y_train)}, test: {np.average(stump.predict(X_test) == y_test)}")

**Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier()
tree.fit(X_train, y_train)
print(f"train: {np.average(tree.predict(X_train) == y_train)}, test: {np.average(tree.predict(X_test) == y_test)}")

**SimpleRandomForest**

In [ ]:
print(f"train: {np.average(randomforest_classifier.predict(X_train) == y_train)}, test: {np.average(randomforest_classifier.predict(X_test) == y_test)}")

**Adaboost**

In [ ]:
print(f"train: {np.average(adaboost_classifier.predict(X_train) == y_train)}, test: {np.average(adaboost_classifier.predict(X_test) == y_test)}")

---

#### P4.3 (1) Analyze and contrast the logistic regression, (simple) random forest, and adaboost based on the *Precision*, *Recall*, and *F1 score* (*5 points*)

Answer:
Logistic Regression shows good recall, indicating it is proficient at capturing a high proportion of actual positives but has a slightly lower precision, which might lead to more false positives. Its F1 score is high, suggesting a balanced approach between precision and recall.
Random Forest outperforms Logistic Regression in all three metrics, showing it not only captures most of the actual positives (high recall) but also maintains high accuracy in its positive predictions (precision). Its higher F1 score indicates that it is the most balanced model among the three, effectively managing the trade-off between precision and recall.
AdaBoost has the lowest precision and recall among the three models. While it still performs adequately, its F1 score is the lowest, indicating that while it manages a reasonable balance between precision and recall, its overall ability to identify and accurately classify positive instances is slightly inferior compared to the other models.
The Random Forest model emerges as the superior model among the three in terms of Precision, Recall, and F1 Score, indicating its robustness and effectiveness in handling the classification task. Logistic Regression, while slightly less effective than Random Forest, offers a commendable balance and could be preferred for simpler models or when interpretability is a priority. AdaBoost, while generally powerful, shows some limitations in this specific context, suggesting that adjustments in its parameters or a review of the feature selection and data preparation stages might be necessary to improve its performance.


In [ ]:
precision_ran = get_precision(randomforest_classifier.predict(X_test), y_test)
recall_ran = get_recall(randomforest_classifier.predict(X_test), y_test)
f1_ran = get_f1_score(randomforest_classifier.predict(X_test), y_test)

precision_ada = get_precision(adaboost_classifier.predict(X_test), y_test)
recall_ada = get_recall(adaboost_classifier.predict(X_test), y_test)
f1_ada = get_f1_score(adaboost_classifier.predict(X_test), y_test)

precision_log = get_precision(regrssor.predict(X_test), y_test)
recall_log = get_recall(regrssor.predict(X_test), y_test)
f1_log = get_f1_score(regrssor.predict(X_test), y_test)

print(f"                    | Precision    | Recall       | F1 score")
print(f"LogisticRegression  | {precision_log:.10f} | {recall_log:.10f} | {f1_log:.10f}")
print(f"RandomForest        | {precision_ran:.10f} | {recall_ran:.10f} | {f1_ran:.10f}")
print(f"AdaBoost            | {precision_ada:.10f} | {recall_ada:.10f} | {f1_ada:.10f}")

---

#### P4.3 (2) Analyze and contrast the logistic regression, (simple) random forest, and adaboost based on the Precision-Recall curve (*5 points*)

Answer:
Analyzing the PR (Precision-Recall) curves, RandomForest demonstrates the best overall performance. It maintains a Precision close to 1.0 up to a Recall of 0.5, and even as Recall increases, the decline in Precision is more gradual compared to other models. The curve is also the smoothest, indicating stable performance across different threshold values.
LogisticRegression shows high initial Precision but exhibits a stepwise pattern in its curve, and notably performs slightly better than RandomForest after a Recall of 0.8. This stepwise pattern suggests that probability estimates are concentrated at certain values.
AdaBoost displays a similar stepwise pattern to LogisticRegression but shows the lowest performance among the three models across most regions. It experiences the most dramatic performance drop in the high Recall region (0.8-1.0), which reveals the limitations of its decision stump-based approach.
Overall, RandomForest provides the most stable and superior performance, while LogisticRegression shows competitive performance in high Recall regions. AdaBoost consistently shows the lowest performance and becomes particularly unreliable when high Recall is required. For practical applications, RandomForest would be the primary choice, with LogisticRegression as a viable alternative when high Recall is necessary.

In [ ]:
pre_ran, rec_ran = plot_precision_recall_curve(randomforest_classifier.predict_proba(X_test), y_test)
pre_ada, rec_ada = plot_precision_recall_curve(adaboost_classifier.predict_proba(X_test), y_test)
pre_log, rec_log = plot_precision_recall_curve(regrssor.predict_proba(X_test), y_test)

# Plot the precision-recall curve
plt.figure(figsize=[9, 8])
plt.plot(rec_ran, pre_ran, marker='.', label='RandomForest')
plt.plot(rec_ada, pre_ada, marker='.', label='AdaBoost')
plt.plot(rec_log, pre_log, marker='.', label='LogisticRegression')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

---

#### P4.4 (1) Run the codes provided to plot decision boundaries and prediction accuracies on test dataset. Analyze and contrast the decision boundaries of the classification methods (logistic regression, bagging decision tree, and boosted decision tree). (*10 points*)  

**Do not use sklearn implementation of Randomforest or Adaboost.**

Answer:
Logistic Regression uses a linear combination of features to calculate a score, which is transformed into probabilities via the logistic (sigmoid) function. This transformation enables logistic regression to establish a decision boundary, effectively distinguishing between two classes in binary classification scenarios. When the features are used as they are, without any transformations, the decision boundary typically appears as a straight line. However, if nonlinear features are derived from the input variables, logistic regression is capable of learning and representing more complex, nonlinear boundaries.

Random Forest builds an ensemble of multiple decision trees, each trained on a bootstrapped subset of the original dataset. It introduces randomness not only through data sampling but also through the feature selection process during the training of each tree. The decision boundary for each individual tree is made up of multiple hyperplanes that segment the feature space, and the overall decision boundary for the model is derived from the majority vote across these trees. In practice, this boundary often appears as a mosaic of hyperplanes—specifically, as vertical lines in scenarios where the decision trees base their splits on single features, leading to a binary decision process.

AdaBoost sequentially trains a series of weak learners (typically simple models like decision stumps), each compensating for the weaknesses of its predecessors by focusing on previously misclassified samples. The decision boundary in AdaBoost results from the cumulative, weighted decision-making of these weak learners. For example, if each weak learner is a decision stump making perpendicular splits to the axis, AdaBoost combines these into a more refined boundary that may take a broader, more rectangular shape. This shape is more extensive compared to the often more fragmented boundaries seen in Random Forest, reflecting the weighted influence of each stump.

In summary, each of these models implements its strategy to form decision boundaries, with logistic regression providing a potentially linear or nonlinear boundary based on the nature of the feature transformation. In contrast, Random Forest creates a complex boundary through the aggregation of simpler ones from individual trees, and AdaBoost develops its boundary by progressively refining it through the sequential training of weak learners. Each model's approach offers distinct advantages and limitations depending on the complexity and type of the data being modeled.

In [ ]:
# prepare dataset.
X,y= make_blobs(n_samples=200, centers=4, cluster_std=2.5, random_state=13)
y[y==1] = 0
y[y==2] = 1
y[y==3] = 1
y[y == 0] = -1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.4, random_state=42)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

names = ["Logisic Regression", "Simple RandomForest", "AdaBoost"]

classifiers = [
    LogisticRegression(),
    SimpleRandomForest(50, max_depth=5),
    AdaBoost(50) # your implementation.
]

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='training samples with class 1', mfc='#FF00FF', mec='k'),
    Line2D([0], [0], marker='o', color='w', label='training samples with class -1', mfc='#00FF00', mec='k'),
    Line2D([0], [0], marker='*', color='w', label='test samples with class 1', mfc='#FF00FF', mec='k', ms=10),
    Line2D([0], [0], marker='*', color='w', label='test samples with class -1', mfc='#00FF00', mec='k', ms=10)
]

figure = plt.figure(figsize=(24, 24))
i = 1

# generate meshgrid for decision boundaries.
x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# plot.
cm = plt.cm.PRGn
cm_bright = ListedColormap(['#FF00FF', '#00FF00'])

ax = plt.subplot(2, 2, 1)
ax.set_title("Input data", size=40)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k', label='train samples')
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=300, marker='*', label='test samples')
ax.set_xlim(xx.min(), xx.max())
ax.set_ylim(yy.min(), yy.max())
ax.legend(handles=legend_elements, ncol=2, loc='lower right', fontsize=20)
ax.set_xticks(())
ax.set_yticks(())

# iterate over classifiers
i += 1
for name, clf in zip(names, classifiers):
    ax = plt.subplot(2, 2, i)
    clf.fit(X_train, y_train)
    score = np.average(clf.predict(X_test) == y_test)

    Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, cmap=cm, alpha=.8)

    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=300, marker='*')

    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.set_xticks(())
    ax.set_yticks(())
    
    ax.set_title(name, size=40)
    ax.text(xx.max() - .3, yy.min() + .3, ('prediction accuracy on test dataset: %.2f' % score).lstrip('0'), size=30, horizontalalignment='right')
    i += 1

plt.tight_layout()
plt.show()

---

#### P4.4 (2) Analyze and contrast the decision boundaries of the classification methods in the aspect of overfitting. Note that we are using more training samples, compared to the previous problem of P4.4(1).   (*5 points*)

Answer:
Logistic Regression For Logistic Regression, the inherent simplicity of the model, where decision boundaries are formed based on linear combinations of input features, can be a drawback when dealing with complex datasets that are not linearly separable. In this scenario, the model tends to underfit, as evidenced by its inability to capture the nuances of the data, leading to a simplistic boundary that does not adequately differentiate between classes.

Simple Random Forest Random Forest leverages an ensemble of decision trees to create a more robust model by averaging out the idiosyncrasies of individual trees, which might overfit on their own. This ensemble effect generally helps in reducing the likelihood of overfitting the entire model. However, overfitting risks still persist, particularly when trees are allowed to grow deep, making the model overly complex for the data at hand. The decision boundaries in Random Forest are typically more confident due to the majority voting system among the trees, which can sometimes lead to overconfidence in predictions, especially on complex datasets.

AdaBoost AdaBoost, another ensemble technique, builds its model sequentially by training weak learners, each focusing more on the samples previously misclassified by its predecessors. This approach allows AdaBoost to develop complex decision boundaries that can effectively handle varied data patterns. However, this sensitivity to misclassified points makes AdaBoost particularly prone to overfitting, especially in the presence of noise within the training data. The cumulative adjustments to weights might amplify the influence of noisy data, skewing the model's ability to generalize.

Model Evaluation By examining the prediction accuracy on a test dataset, it's possible to gauge the extent of overfitting. High accuracy might indicate good generalization, whereas a significant discrepancy between training and testing performance often suggests overfitting. Such analysis is crucial in validating the appropriateness of the decision boundary shaped by each model and in assessing their respective risks and strengths in real-world scenarios.

In [ ]:
# Increase the number of training samples.
X,y= make_blobs(n_samples=400, centers=4, cluster_std=4, random_state=13)
y[y==1] = 0
y[y==2] = 1
y[y==3] = 1
y[y == 0] = -1

# Test samples are the same as previous ones.
X_train, _, y_train, _ = train_test_split(X, y, test_size=.4, random_state=42)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

names = ["Logisic Regression", "Simple RandomForest", "AdaBoost"]

classifiers = [
    LogisticRegression(),
    SimpleRandomForest(50, max_depth=5),
    AdaBoost(50) # your implementation.
]

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='training samples with class 1', mfc='#FF00FF', mec='k'),
    Line2D([0], [0], marker='o', color='w', label='training samples with class -1', mfc='#00FF00', mec='k'),
    Line2D([0], [0], marker='*', color='w', label='test samples with class 1', mfc='#FF00FF', mec='k', ms=10),
    Line2D([0], [0], marker='*', color='w', label='test samples with class -1', mfc='#00FF00', mec='k', ms=10)
]

figure = plt.figure(figsize=(24, 24))
i = 1

# generate meshgrid for decision boundaries.
x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# plot.
cm = plt.cm.PRGn
cm_bright = ListedColormap(['#FF00FF', '#00FF00'])

ax = plt.subplot(2, 2, 1)
ax.set_title("Input data", size=40)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k', label='train samples')
ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=300, marker='*', label='test samples')
ax.set_xlim(xx.min(), xx.max())
ax.set_ylim(yy.min(), yy.max())
ax.legend(handles=legend_elements, ncol=2, loc='lower right', fontsize=20)
ax.set_xticks(())
ax.set_yticks(())

# iterate over classifiers
i += 1
for name, clf in zip(names, classifiers):
    ax = plt.subplot(2, 2, i)
    clf.fit(X_train, y_train)
    score = np.average(clf.predict(X_test) == y_test)

    Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, cmap=cm, alpha=.8)

    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=300, marker='*')

    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.set_xticks(())
    ax.set_yticks(())
    
    ax.set_title(name, size=40)
    ax.text(xx.max() - .3, yy.min() + .3, ('prediction accuracy on test dataset: %.2f' % score).lstrip('0'), size=30, horizontalalignment='right')
    i += 1

plt.tight_layout()
plt.show()

---